In [19]:
import pyspark.sql.functions as F
from pyiceberg.catalog import load_catalog
from pyspark.sql.types import *

from utils import *

catalog = load_catalog(
    CATALOG_NAME,
    type="sql",
    uri=CATALOG_URI,
    warehouse=WAREHOUSE_PATH,
)

In [2]:
spark = get_spark()

26/04/11 20:10:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.8
spark.jars: /opt/spark-jars/iceberg-spark-runtime-3.5_2.12-1.10.1.jar,/opt/spark-jars/postgresql-42.7.5.jar


In [3]:
RAW_TABLE = f"{CATALOG_NAME}.{CATALOG_DB}.raw_logs"
FORMATTED_TABLE = f"{CATALOG_NAME}.{CATALOG_DB}.formatted_speed_test_logs"

In [4]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG_NAME}.{CATALOG_DB}")

26/04/11 20:10:30 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1


""


In [11]:
(
    spark.readStream.format('json').schema(SCHEMA_LOGS).load(LOGS_PATH)
    .writeStream.format("iceberg")
    .outputMode("append")
    .trigger(processingTime="1 minute")
    .option("checkpointLocation", checkpoint_path("raw_logs"))
    .toTable(RAW_TABLE)
)

26/04/11 20:13:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [12]:
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{CATALOG_DB}").show(truncate=False)

+----------+-------------------------+-----------+
|namespace |tableName                |isTemporary|
+----------+-------------------------+-----------+
|speed_test|formatted_speed_test_logs|false      |
|speed_test|raw_logs                 |false      |
+----------+-------------------------+-----------+



In [13]:
def formatting_raw_logs():

    sdf = spark.readStream.format("iceberg").table(RAW_TABLE)

    # Convert timestamp string to timestamp type
    ts = F.to_timestamp("timestamp")
    hour_col = F.hour(ts)

    (
        sdf.select(
            # Flatten nested result ID
            F.col("result.id").alias("id"),
            # ISP information
            F.col("isp"),
            # Timestamp and date/time components
            ts.alias("timestamp"),
            F.date_format(ts, "yyyy-MM-dd").alias("date"),
            F.date_format(ts, "HH:mm:ss").alias("time"),
            F.year(ts).alias("year"),
            F.month(ts).alias("month"),
            F.dayofmonth(ts).alias("day"),
            hour_col.alias("hour"),
            F.minute(ts).alias("minute"),
            F.second(ts).alias("second"),
            # Categorize time of day
            F.when((hour_col >= 5) & (hour_col < 12), "Morning")
            .when((hour_col >= 12) & (hour_col < 17), "Afternoon")
            .when((hour_col >= 17) & (hour_col < 21), "Evening")
            .otherwise("Night")
            .alias("part_of_the_day"),
            # Day of week (1=Sunday, 2=Monday, ..., 7=Saturday)
            F.dayofweek(ts).cast("int").alias("dayofweek"),
            # Convert bytes to Megabytes for readability
            (F.col("download.bytes") / F.lit(1000000.0)).alias("download_Mbytes"),
            (F.col("upload.bytes") / F.lit(1000000.0)).alias("upload_Mbytes"),
        )
        .writeStream.format("iceberg")
        .option("checkpointLocation", checkpoint_path("formatted_logs"))
        .toTable(FORMATTED_TABLE)
    )


formatting_raw_logs()

26/04/11 20:13:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [31]:
streaming_logs_sdf = spark.read.table(FORMATTED_TABLE)

In [17]:
def create_dayofweek_names():
    """
    Create a reference DataFrame with day of week names in Spanish and English
    
    This helper function generates a lookup table to map day of week numbers
    (1-7) to their names in both Spanish and English languages.
    
    Returns:
        DataFrame: Reference table with columns:
            - dayofweek (int): Day number (1=Sunday, 2=Monday, ..., 7=Saturday)
            - esp (string): Spanish day name
            - eng (string): English day name
            
    Example:
        >>> create_dayofweek_names().show()
        +---------+----------+---------+
        |dayofweek|esp       |eng      |
        +---------+----------+---------+
        |1        |Domingo   |Sunday   |
        |2        |Lunes     |Monday   |
        ...
    """
    # Day of week numbers (1=Sunday, 2=Monday, ..., 7=Saturday)
    index = [1, 2, 3, 4, 5, 6, 7]
    
    # Spanish day names
    esp = ["Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]
    
    # English day names
    eng = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]

    # Combine into rows
    rows = list(zip(index, esp, eng))

    # Create DataFrame with explicit schema
    return spark.createDataFrame(rows, ["dayofweek", "esp", "eng"]).withColumn(
        "dayofweek", F.col("dayofweek").cast(IntegerType())
    )



def agg_speed_test(sdf, groupBy):
    """
    Aggregate speed test metrics by specified grouping columns
    
    This helper function performs standard aggregations on speed test data:
    - Average download speed (Mbytes)
    - Average upload speed (Mbytes)
    - Count of tests
    
    Args:
        sdf (DataFrame): Input DataFrame with speed test data
        groupBy (list or str): Column(s) to group by
        
    Returns:
        DataFrame: Aggregated DataFrame with avg_download_speed, avg_upload_speed, count_of_test
        
    Example:
        >>> agg_speed_test(df, ["date", "hour"])
        >>> agg_speed_test(df, "dayofweek")
    """
    return sdf.groupBy(groupBy).agg(
        F.mean("download_Mbytes").alias("avg_download_speed"),
        F.mean("upload_Mbytes").alias("avg_upload_speed"),
        F.count("id").alias("count_of_test"),
    )




In [32]:
def update_agg_by_day_of_week():
    """
    Average download speed by day of week - Gold layer
    
    This streaming table aggregates speed test data by day of week
    and joins with day name reference data to provide human-readable
    day names in both Spanish and English.
    
    Useful for identifying patterns in internet performance across
    different days of the week (e.g., slower on weekends vs weekdays).
    
    Returns:
        DataFrame: Streaming DataFrame with aggregated metrics by day of week
        
    Output Columns:
        - dayofweek: Day number (1-7)
        - avg_download_speed: Average download speed in Mbytes
        - avg_upload_speed: Average upload speed in Mbytes
        - count_of_test: Number of tests for that day of week
        - esp: Spanish day name
        - eng: English day name
        
    Example Output:
        dayofweek | avg_download_speed | count_of_test | esp     | eng
        ----------|-------------------|---------------|---------|--------
        1         | 245.3             | 42            | Domingo | Sunday
        2         | 287.1             | 38            | Lunes   | Monday
    """
    
    # Read from silver streaming table
    streaming_logs_sdf = spark.read.table(FORMATTED_TABLE)
    # Aggregate by day of week
    aggregated_sdf = agg_speed_test(streaming_logs_sdf, "dayofweek").join(
        F.broadcast(create_dayofweek_names()), on="dayofweek", how="right"
    )


In [28]:
streaming_logs_sdf = spark.readStream.table(FORMATTED_TABLE)

26/04/11 20:18:32 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for right outer join.
26/04/11 20:18:33 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for right outer join.
26/04/11 20:18:33 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for right outer join.
26/04/11 20:18:33 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build right for right outer join.


dayofweek,avg_download_speed,avg_upload_speed,count_of_test,esp,eng
1,0.0,0.0,0,Domingo,Sunday
2,0.0,0.0,0,Lunes,Monday
3,0.0,0.0,0,Martes,Tuesday
4,0.0,0.0,0,Miércoles,Wednesday
5,0.0,0.0,0,Jueves,Thursday
6,0.0,0.0,0,Viernes,Friday
7,235.70809824999995,194.17168981818176,88,Sábado,Saturday
